In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

def run(command):
    print('$', ' '.join(str(x) for x in command), flush=True)
    subprocess.run([str(x) for x in command], check=True)

run([sys.executable, '-m', 'pip', 'install', '-q',
     'pymupdf>=1.25.5,<1.26', 'pillow>=10,<12',
     'transformers>=4.40,<5', 'sentencepiece>=0.2,<1',
     'opencv-python-headless>=4.10,<5', 'pydantic>=2.7,<3',
     'pydantic-settings>=2.2,<3', 'pyyaml>=6,<7'])

import torch
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required; do not run this on CPU.')
capability = torch.cuda.get_device_capability(0)
if capability < (7, 0):
    raise RuntimeError(f'Unsupported GPU sm_{capability[0]}{capability[1]}; select a T4 or newer.')
print('GPU:', torch.cuda.get_device_name(0), f'sm_{capability[0]}{capability[1]}', flush=True)

work = Path('/kaggle/working')
repo = work / 'doc-agent-G07'
if not repo.exists():
    run(['git', 'clone', '--branch', 'a2/trocr-layout-comparison', '--depth', '1',
         'https://github.com/smammahdi/doc-agent-G07.git', repo])
else:
    run(['git', '-C', repo, 'fetch', 'origin', 'a2/trocr-layout-comparison', '--depth', '1'])
    run(['git', '-C', repo, 'checkout', '--force', 'FETCH_HEAD'])
run(['git', '-C', repo, 'rev-parse', 'HEAD'])

wrapper = repo / 'extras' / 'ocr_research' / 'run_kaggle_layout_trocr.py'
output = work / 'trocr_layout_outputs'
cache = work / 'trocr_page_cache'
run([sys.executable, wrapper, '--input-root', '/kaggle/input',
     '--output-root', output, '--cache-root', cache, '--device', 'cuda',
     '--batch-size', '8', '--max-length', '64', '--dpi', '300'])
archive = shutil.make_archive(str(work / 'trocr_layout_outputs'), 'zip', output)
print('canonical output:', output)
print('downloadable archive:', archive)
